# ANPR End-to-End (Colab Notebook)

This notebook creates a venv, installs dependencies, downloads dataset sources, optionally trains, and runs the app.

In [10]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/ayannotfound/ANPR.git'
DEFAULT_CLONE_DIR = Path('/content/ANPR')

def looks_like_repo(path: Path) -> bool:
    return (path / 'requirements.txt').exists() and (path / 'main.py').exists()

def resolve_repo_root() -> Path | None:
    cwd = Path.cwd()
    candidates = [cwd, DEFAULT_CLONE_DIR]
    candidates.extend(cwd.parents)
    for c in candidates:
        if looks_like_repo(c):
            return c
    return None

repo_root = resolve_repo_root()
if repo_root is None:
    print('Repo not found in kernel filesystem. Cloning now...')
    if DEFAULT_CLONE_DIR.exists() and not looks_like_repo(DEFAULT_CLONE_DIR):
        subprocess.run(['rm', '-rf', str(DEFAULT_CLONE_DIR)], check=False)
    if not DEFAULT_CLONE_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(DEFAULT_CLONE_DIR)], check=True)
    repo_root = DEFAULT_CLONE_DIR

if not looks_like_repo(repo_root):
    raise FileNotFoundError('Repository still not found after clone attempt.')

os.chdir(repo_root)
print('Repo root:', repo_root)

venv_dir = Path('.venv')
venv_python = venv_dir / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')

if not venv_python.exists():
    try:
        subprocess.run([sys.executable, '-m', 'venv', str(venv_dir)], check=True)
    except subprocess.CalledProcessError:
        print('Venv creation failed; using current interpreter.')

selected_python = str(venv_python) if venv_python.exists() else sys.executable
os.environ['ANPR_PYTHON'] = selected_python

# Best-effort pip bootstrap for the selected interpreter.
subprocess.run([selected_python, '-m', 'ensurepip', '--upgrade'], check=False)
subprocess.run([selected_python, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)

req_cmd = [selected_python, '-m', 'pip', 'install', '-r', 'requirements.txt']
req_proc = subprocess.run(req_cmd, text=True, capture_output=True)
if req_proc.returncode != 0:
    print('\nrequirements install failed. Last stderr lines:')
    print(req_proc.stderr[-2000:])
    raise subprocess.CalledProcessError(req_proc.returncode, req_cmd)

print('Python for notebook steps:', os.environ['ANPR_PYTHON'])

Repo not found in kernel filesystem. Cloning now...
Repo root: /content/ANPR
Venv creation failed; using current interpreter.


CalledProcessError: Command '['.venv/bin/python', '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel']' returned non-zero exit status 1.

## Dataset Key at Runtime

Roboflow API key link: https://app.roboflow.com/settings/api

In [ ]:
import os
from getpass import getpass

rf_key = getpass('Enter ROBOFLOW_API_KEY (leave blank to skip Roboflow source): ').strip()
if rf_key:
    os.environ['ROBOFLOW_API_KEY'] = rf_key
    print('Roboflow key saved in this runtime session.')
else:
    print('No key entered. download_dataset.py will proceed with HF only.')

In [ ]:
# Inline dataset build logic (mirrors download_dataset.py main path)
import os
import sys
from pathlib import Path
from download_dataset import build_dataset

if not (Path('download_dataset.py').exists() and Path('main.py').exists()):
    raise FileNotFoundError('Run Cell 2 first to set repo root correctly.')

selected_python = os.environ.get('ANPR_PYTHON', sys.executable)
print('Using interpreter:', selected_python)

include_hf = True
include_roboflow = True
roboflow_version = 1
clean_temp = True
roboflow_api_key = os.getenv('ROBOFLOW_API_KEY')

print('=' * 72)
print('ANPR Dataset Builder')
print('=' * 72)
print(f'Sources: HF={include_hf}, Roboflow Indian={include_roboflow}')

yaml_path = build_dataset(
    include_hf=include_hf,
    include_roboflow=include_roboflow,
    roboflow_api_key=roboflow_api_key,
    roboflow_version=roboflow_version,
    clean_temp=clean_temp,
)

print('\nUse this for training:')
print(f'  python train.py  # DATA_YAML already points to: {yaml_path.as_posix()}')

## Training (Only If Needed)

Training is slow and expensive. Keep this disabled unless you need a better detector.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path('train.py').exists():
    raise FileNotFoundError('Run Cell 2 first to set repo root correctly.')

TRAIN_IF_NEEDED = False
selected_python = os.environ.get('ANPR_PYTHON', sys.executable)

if TRAIN_IF_NEEDED:
    subprocess.run([selected_python, 'train.py'], check=True)
else:
    print('Training skipped (default).')

In [ ]:
from pathlib import Path
from webapp import create_app, run_server

if not Path('main.py').exists():
    raise FileNotFoundError('Run Cell 2 first to set repo root correctly.')

app = create_app()
run_server()